# HiST-MAT-X Paper 2: spatial analysis of area-level transfer gains (Session 7, CPU only)
**Settings:** Accelerator **None** (CPU is enough), Internet **On**.
**Input:** add the output of the last main notebook version (Session 6), the same input you used before.
**Run:** Save Version → Save & Run All. It takes about 5–15 minutes.

The notebook does not train anything. For each city it:
1. rebuilds the area index exactly as the main notebook did;
2. reads the per-area test log-loss saved for every run;
3. downloads the official area boundaries (with a fallback to area centroids from the incident coordinates);
4. computes global Moran's I of the per-area gain (queen contiguity and 5-nearest-neighbour weights, 9,999 permutations),
   also after removing the effect of area volume, and draws the maps.

At the end it writes `spatial_outputs.zip`. Download that file and upload it in the chat.

In [ ]:
import os, re, glob, json, math, shutil, zipfile, warnings, urllib.request
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib, matplotlib.ticker; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SP_INPUT_ROOT", "/kaggle/input")
OUT = Path(os.environ.get("SP_OUT", "/kaggle/working/spatial_outputs")); OUT.mkdir(parents=True, exist_ok=True)
LOCAL_GEO = os.environ.get("SP_LOCAL_GEO", "")          # only for offline testing
N_PERM, K_NN, MIN_N, SEEDS = 9999, 5, 20, range(5)
CITIES = ["CHI", "NYC", "SF"]; CN = {"CHI": "Chicago", "NYC": "New York", "SF": "San Francisco"}
UTM = {"CHI": 32616, "NYC": 32618, "SF": 32610}
TEST_START, TEST_END = pd.Timestamp("2025-12-15"), pd.Timestamp("2026-06-15")   # weeks 363-388 (end exclusive)
LOG = []
import time as _t; _T0 = _t.time()
def log(*a):
    s = f"[{_t.time() - _T0:5.0f} s] " + " ".join(str(x) for x in a); print(s, flush=True); LOG.append(s)

## 1. Find the runs and the raw data in the input

In [ ]:
import time; T0 = time.time()
def find_dir(pattern, base):
    c = glob.glob(f"{base}/**/{pattern}", recursive=True)
    return max(c, key=lambda p: len(os.listdir(Path(p).parent))) if c else None
log("searching the input folders ...")
RUNS = find_dir("runs/main__type_CHI_8_GBDT_0.npz", INPUT_ROOT)
DATA = find_dir("data/CHI.csv", INPUT_ROOT)
log(f"  folders: runs={RUNS} data={DATA} ({time.time()-T0:.0f} s)")
if RUNS is None or DATA is None:                 # fall back to the zip, extracting only the files needed
    zips = sorted(glob.glob(f"{INPUT_ROOT}/**/histmatx_outputs*.zip", recursive=True), key=os.path.getsize, reverse=True)
    log("  looking inside zip files:", zips)
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            names = [n for n in zf.namelist() if (RUNS is None and re.search(r"runs/main__(type|slot)_(CHI|NYC|SF)_(4|8|52)_(GBDT|HMX-pre|FTT-pre)_\d\.npz$", n))
                     or (DATA is None and re.search(r"data/(CHI|NYC|SF)\.csv$", n))]
            log(f"  {z}: extracting {len(names)} files"); zf.extractall("/tmp/sp_zip", members=names)
        RUNS = RUNS or find_dir("runs/main__type_CHI_8_GBDT_0.npz", "/tmp/sp_zip")
        DATA = DATA or find_dir("data/CHI.csv", "/tmp/sp_zip")
        if RUNS and DATA: break
assert RUNS, "No 'runs' folder found. Add the Session 6 notebook output as input."
assert DATA, "No 'data/CHI.csv' found. Add the full Session 6 notebook output (not only the zip) as input."
RUNS, DATA = Path(RUNS).parent, Path(DATA).parent
log("runs:", RUNS); log("data:", DATA)

## 2. Rebuild each city's area index exactly as the main notebook did
(same de-duplication, date filter, offence mapping and sorted factorisation).

In [ ]:
TYPES = ["ASSAULT", "ROBBERY", "SEX_OFFENSE", "THEFT", "BURGLARY", "VEHICLE_THEFT", "DAMAGE", "FRAUD", "DRUGS", "WEAPONS"]
TYPE_RULES = [(r"MOTOR VEHICLE|VEHICLE THEFT|STOLEN VEHICLE", "VEHICLE_THEFT"), (r"ROBBERY", "ROBBERY"), (r"BURGLAR", "BURGLARY"),
              (r"SEX|RAPE", "SEX_OFFENSE"), (r"WEAPON", "WEAPONS"), (r"NARCOTIC|DRUG", "DRUGS"),
              (r"FRAUD|DECEPTIVE|FORGERY|EMBEZZLE|COUNTERFEIT", "FRAUD"), (r"CRIMINAL DAMAGE|MISCHIEF|VANDAL", "DAMAGE"),
              (r"ASSAULT|BATTERY", "ASSAULT"), (r"THEFT|LARCENY", "THEFT")]
def _clean(s): return s.map(lambda x: "" if pd.isna(x) else str(x)).astype(object).str.upper().str.strip()
def map_type(series):
    out = {}
    for v in pd.Series(_clean(series).unique()):
        lab = None
        if v not in ("NAN", "", "NONE", "(NULL)", "UNKNOWN", "N/A"):
            for pat, name in TYPE_RULES:
                if re.search(pat, v): lab = name; break
        out[v] = lab
    return _clean(series).map(out)
def parse_dt(s):
    try: return pd.to_datetime(s, format="ISO8601", errors="coerce")
    except Exception: return pd.to_datetime(s, errors="coerce")

def areas_of(city):
    log(f"reading {city} raw data ..."); r = pd.read_csv(DATA / f"{city}.csv", low_memory=False)
    if city == "CHI":
        if "id" in r: r = r.drop_duplicates("id")
        dt = parse_dt(r["date"])
        if dt.isna().mean() > .5: dt = pd.to_datetime(r["date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
        typ, area = r["primary_type"], pd.to_numeric(r["community_area"], errors="coerce"); area = area.where(area > 0)
    elif city == "NYC":
        if "cmplnt_num" in r: r = r.drop_duplicates("cmplnt_num", keep="last")
        dt = pd.to_datetime(r["cmplnt_fr_dt"].astype(str).str[:10], format="%Y-%m-%d", errors="coerce")
        typ, area = r["ofns_desc"], pd.to_numeric(r["addr_pct_cd"], errors="coerce")
    else:
        if "incident_id" in r: r = r.drop_duplicates("incident_id")
        dt = parse_dt(r["incident_datetime"]); typ = r["incident_category"]
        area = r["analysis_neighborhood"].where(r["analysis_neighborhood"].astype(str).str.upper().isin(["NAN", "NONE", ""]) == False)
    d = pd.DataFrame(dict(dt=dt.values, type=map_type(typ).values, area_raw=area.values,
                          lat=pd.to_numeric(r["latitude"], errors="coerce").values, lon=pd.to_numeric(r["longitude"], errors="coerce").values))
    d = d[d.dt.notna() & (d.dt >= pd.Timestamp("2019-01-01")) & d.type.notna() & pd.notna(d.area_raw)].copy()
    codes, uniq = pd.factorize(d.area_raw.astype(str), sort=True); d["area"] = codes
    ok = d.lat.between(d.lat.quantile(.001) - .05, d.lat.quantile(.999) + .05) & d.lon.between(d.lon.quantile(.001) - .05, d.lon.quantile(.999) + .05) & (d.lat.abs() > 1)
    cen = d[ok].groupby("area")[["lat", "lon"]].median().reindex(range(len(uniq)))
    te = d[(d.dt >= TEST_START) & (d.dt < TEST_END)]
    n_test = np.bincount(te.area.values, minlength=len(uniq))
    return pd.DataFrame(dict(area=range(len(uniq)), key=list(map(str, uniq)), lat=cen.lat.values, lon=cen.lon.values, n_test_raw=n_test))

def load(task, city, bud, model, seed):
    p = RUNS / (re.sub(r"[^A-Za-z0-9_.-]", "_", f"main__{task}|{city}|{bud}|{model}|{seed}") + ".npz")
    return dict(np.load(p)) if p.exists() else None
def area_ll(task, city, bud, model):
    R = [load(task, city, bud, model, s) for s in SEEDS]; R = [x for x in R if x is not None]
    if not R: return None, None
    return np.mean([x["nll_a"] / np.maximum(x["n_a"], 1) for x in R], 0), R[0]["n_a"]

A = {}
for c in CITIES:
    a = areas_of(c); _, n_a = area_ll("type", c, "8", "GBDT")
    log(f"{c}: {len(a)} areas rebuilt, {len(n_a)} areas in the runs")
    assert len(a) == len(n_a), f"{c}: area count mismatch – stop and report this message in the chat"
    rho = pd.Series(a.n_test_raw).corr(pd.Series(n_a), method="spearman")
    log(f"{c}: Spearman(raw test count, run test count) = {rho:.4f}  (should be > 0.98)")
    assert rho > .95, f"{c}: area order does not match – stop and report this message in the chat"
    a["n_test"] = n_a; A[c] = a

## 3. Official boundaries (with centroid fallback)

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
GEO = {"CHI": ("data.cityofchicago.org", "igwz-8jzy"), "NYC": ("data.cityofnewyork.us", "y76i-bdw7"), "SF": ("data.sfgov.org", "j2bu-swwd")}
def fetch_geo(city):
    if LOCAL_GEO and Path(LOCAL_GEO, f"{city}.geojson").exists(): return gpd.read_file(Path(LOCAL_GEO, f"{city}.geojson")), "local file"
    host, ds = GEO[city]
    for url in [f"https://{host}/resource/{ds}.geojson?$limit=5000", f"https://{host}/api/geospatial/{ds}?method=export&format=GeoJSON",
                f"https://{host}/api/v3/views/{ds}/query.geojson?accessType=DOWNLOAD"]:
        try:
            p = OUT / f"boundaries_{city}.geojson"
            with urllib.request.urlopen(url, timeout=60) as r: p.write_bytes(r.read())
            g = gpd.read_file(p)
            if len(g) > 10: return g, url
        except Exception as e: log(f"  {city}: {url} failed: {e!r}"[:200])
    return None, None
def geo_key(city, g):
    cols = {c.lower(): c for c in g.columns}
    if city == "CHI":
        f = next(cols[c] for c in ["area_numbe", "area_num_1", "area_number", "community_area"] if c in cols)
        return g[f].map(lambda v: str(float(v)))
    if city == "NYC":
        f = next(cols[c] for c in ["precinct", "pct", "addr_pct_cd"] if c in cols); return g[f].map(lambda v: str(float(v)))
    f = next(cols[c] for c in ["nhood", "analysis_neighborhood", "neighborhood", "name"] if c in cols); return g[f].astype(str)

G, SRC = {}, {}
for c in CITIES:
    log(f"{c}: downloading boundaries ..."); g, src = fetch_geo(c)
    if g is not None:
        g = g.set_crs(4326, allow_override=True) if g.crs is None else g
        g["key"] = geo_key(c, g).values
        if c == "SF":                                       # tolerate case/space differences in names
            norm = lambda s: re.sub(r"\s+", " ", s.strip().lower())
            m = {norm(k): k for k in A[c].key}; g["key"] = g.key.map(lambda k: m.get(norm(k), k))
        g = g.dissolve("key").reset_index()
        miss_g = sorted(set(A[c].key) - set(g.key)); miss_d = sorted(set(g.key) - set(A[c].key))
        log(f"{c}: boundaries from {src}: {len(g)} polygons; areas without polygon: {miss_g}; polygons without data: {miss_d}")
        if len(miss_g) > 0.1 * len(A[c]): log(f"{c}: too many unmatched areas, using centroids only"); g = None
    if g is None:
        g = gpd.GeoDataFrame(A[c][["key"]], geometry=[Point(xy) for xy in zip(A[c].lon, A[c].lat)], crs=4326); src = "centroids"
    G[c], SRC[c] = g.to_crs(UTM[c]), src

## 4. Gains and Moran's I

In [ ]:
rng = np.random.default_rng(2026)
def weights_knn(xy, k=K_NN):
    D = np.sqrt(((xy[:, None, :] - xy[None, :, :]) ** 2).sum(-1)); np.fill_diagonal(D, np.inf)
    W = np.zeros_like(D); idx = np.argsort(D, 1)[:, :k]
    for i, js in enumerate(idx): W[i, js] = 1
    return W
def weights_queen(geoms):
    n = len(geoms); W = np.zeros((n, n)); b = [g.buffer(1.0) for g in geoms]       # 1 m tolerance for slivers
    for i in range(n):
        for j in range(i + 1, n):
            if b[i].intersects(b[j]): W[i, j] = W[j, i] = 1
    return W
def moran(z, W, nperm=N_PERM):
    keep = W.sum(1) > 0; z, W = z[keep], W[np.ix_(keep, keep)]
    W = W / W.sum(1, keepdims=True); z = z - z.mean(); n = len(z)
    I = n / W.sum() * (z @ W @ z) / (z @ z)
    sims = np.array([(lambda p: n / W.sum() * (p @ W @ p) / (p @ p))(rng.permutation(z)) for _ in range(nperm)])
    p = (1 + np.sum(np.abs(sims - sims.mean()) >= abs(I - sims.mean()))) / (nperm + 1)
    return I, -1 / (n - 1), p, n
def residual(gain, n):
    X = np.c_[np.ones(len(n)), np.log(n)]; b = np.linalg.lstsq(X, gain, rcond=None)[0]; return gain - X @ b

ROWS, GAIN = [], {}
CMP = [("type", "8", "HMX-pre", "GBDT", "main"), ("type", "4", "HMX-pre", "GBDT", "robust"), ("type", "8", "FTT-pre", "GBDT", "robust"),
       ("slot", "8", "HMX-pre", "GBDT", "robust"), ("type", "52", "HMX-pre", "GBDT", "robust")]
for c in CITIES:
    a = A[c].copy(); g = G[c].set_index("key")
    has = a.key.isin(g.index).values
    for task, bud, mA, mB, role in CMP:
        la, na = area_ll(task, c, bud, mA); lb, nb = area_ll(task, c, bud, mB)
        if la is None or lb is None: log(f"{c} {task} {bud} {mA}: missing runs"); continue
        gain = lb - la                                         # positive = model A better
        sel = (na >= MIN_N) & has
        sub = a[sel].reset_index(drop=True); gs = gain[sel]; nn = na[sel].astype(float)
        geoms = g.loc[sub.key].geometry.values
        xy = np.c_[[gm.centroid.x for gm in geoms], [gm.centroid.y for gm in geoms]]
        Ws = {"knn5": weights_knn(xy)}
        if SRC[c] != "centroids": Ws["queen"] = weights_queen(geoms)
        for wname, W in Ws.items():
            for target, z in [("gain", gs), ("gain_resid_logvol", residual(gs, nn))]:
                I, EI, p, n = moran(z.astype(float), W.copy())
                ROWS.append(dict(city=c, task=task, weeks=bud, model_A=mA, model_B=mB, role=role, weights=wname, variable=target,
                                 n_areas=n, isolates=int((W.sum(1) == 0).sum()), moran_I=I, expected_I=EI, p_perm=p))
        if role == "main" or (task, bud, mA) == ("slot", "8", "HMX-pre"):
            GAIN[(c, task, bud, mA)] = pd.DataFrame(dict(key=sub.key, n_test=nn, gain=gs))
MO = pd.DataFrame(ROWS); MO.to_csv(OUT / "moran.csv", index=False)
pd.set_option("display.width", 200); print(MO.round(4).to_string())

# area-level table for the paper / ESM
rows = []
for (c, task, bud, m), df in GAIN.items():
    df = df.assign(city=c, task=task, weeks=bud, model=m); rows.append(df)
pd.concat(rows).to_csv(OUT / "area_gains.csv", index=False)
json.dump(SRC, open(OUT / "boundary_sources.json", "w"), indent=1)

## 5. Maps (Fig. 6): crime-type gain of HiST-MAT-X over GBDT at eight weeks

In [ ]:
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 8, "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.dpi": 600})
CM = LinearSegmentedColormap.from_list("div", ["#eb6834", "#f7f7f5", "#2a78d6"])
vals = np.concatenate([GAIN[(c, "type", "8", "HMX-pre")].gain.values for c in CITIES])
lim = np.quantile(np.abs(vals), .95)
fig, axes = plt.subplots(1, 3, figsize=(174 / 25.4, 76 / 25.4))
for ax, c, s in zip(axes, CITIES, "abc"):
    g = G[c].copy(); df = GAIN[(c, "type", "8", "HMX-pre")].set_index("key")
    g["gain"] = g.key.map(df.gain)
    if SRC[c] == "centroids":
        g.plot(ax=ax, column="gain", cmap=CM, norm=TwoSlopeNorm(0, -lim, lim), markersize=18, edgecolor="#52514e", linewidth=.3,
               missing_kwds=dict(color="#d9d8d4", markersize=6))
    else:
        g.plot(ax=ax, column="gain", cmap=CM, norm=TwoSlopeNorm(0, -lim, lim), edgecolor="#52514e", linewidth=.25,
               missing_kwds=dict(color="#d9d8d4", hatch="////", edgecolor="#9a9994", linewidth=.2))
    r = MO[(MO.city == c) & (MO.role == "main") & (MO.variable == "gain")]
    r = r[r.weights == ("queen" if (r.weights == "queen").any() else "knn5")].iloc[0]
    ax.set_axis_off(); ax.set_anchor("N")
    ax.set_title(f"({s}) {CN[c]}\nMoran's I = {r.moran_I:.2f}, p = {r.p_perm:.3f}", loc="left", fontsize=8, color="#0b0b0b")
sm = plt.cm.ScalarMappable(cmap=CM, norm=TwoSlopeNorm(0, -lim, lim))
cb = fig.colorbar(sm, ax=axes, orientation="horizontal", fraction=.05, pad=.04, aspect=40, extend="both")
cb.set_label("log-loss gain over GBDT (blue: HiST-MAT-X better; orange: worse; grey hatched: < 20 test incidents)", fontsize=7)
cb.ax.tick_params(labelsize=7); cb.locator = matplotlib.ticker.MaxNLocator(5); cb.update_ticks()
for ext in ["pdf", "eps", "png"]: fig.savefig(OUT / f"Fig6_maps.{ext}", bbox_inches="tight")
plt.show()

## 6. Pack everything

In [ ]:
(OUT / "log.txt").write_text("\n".join(LOG))
shutil.make_archive("/kaggle/working/spatial_outputs" if OUT.parent == Path("/kaggle/working") else str(OUT), "zip", OUT)
print("done – download spatial_outputs.zip from the Output panel")